# 51 Batch mesh statistics + bidirectional vertex sweep

Runs the anonymization pipeline on every valid thesis subject and collects two families of mesh-level quantities Chapter 4 cites:

1. **Mesh statistics (§4.3, Table 4.4):** vertex/face counts before and after deletion, percentage of vertices removed, cap-boundary height, and degenerate-face percentage on the anonymized mesh.
2. **Bidirectional vertex-identity sweep (§4.4.2):** the forward sweep -- query each anonymized vertex against a KDTree of the input surface -- and the reverse sweep -- query each input vertex that is non-masked AND incident to at least one fully non-masked face. Both sweeps must return a maximum distance of 0 and zero mismatches; the anonymization operator deletes vertices but does not move surviving ones. The boundary-orphan diagnostic, the count of non-masked vertices whose every incident face was discarded because of a single masked neighbour, is reported alongside.

**Cohort.** Iterates over the eleven valid thesis subjects: the optode-cap cohort S1--S7 (Subject 16--22) and the bare-cap cohort S8--S11 (Subject 12--15). Subject 11 is excluded -- the raw scan is unusable due to an acquisition-side defect. The CSV carries the shared `s_id` and `optode` columns so the LaTeX tables can render cohort membership directly.

Output: `thesis_results_out/batch_validation.csv`.

**Prerequisite.** Each subject needs a `Subject{N}_anon_landmarks.tsv` sidecar, written by notebook 48. Subjects without the sidecar are skipped with a warning.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report, run_pipeline,
    is_optode, s_id,
)

import numpy as np
import pandas as pd
from scipy.spatial import KDTree

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

## 1. Which subjects are ready?

In [2]:
ready = available_subjects()

print(f'Ready: {ready}')

missing = missing_report()

if missing:

    print('Missing files:')

    for n, items in missing.items():

        print(f'  Subject{n}: {items}')

Ready: [16, 17, 18, 19, 20, 21, 22, 12, 13, 14, 15]


## 2. Run pipeline per subject, collect mesh statistics

For each ready subject we load the raw scan and landmarks, run the pipeline, and record mesh statistics on the CTF-aligned head (`surface_ctf`) and the anonymized mesh (`surface_anon_ctf`). No validator call; just direct counts.

In [3]:
rows = []
for n in ready:
    print(f'--- {s_id(n)} (Subject{n}) ---')
    surface_raw = load_raw(n)
    landmarks_raw = load_landmarks(n)
    art = run_pipeline(surface_raw, landmarks_raw, subject=n)

    n_head = art.surface_ctf.nvertices
    n_faces_head = art.surface_ctf.nfaces
    n_anon = art.surface_anon_ctf.nvertices
    n_faces_anon = art.surface_anon_ctf.nfaces
    mask_size = int(art.mask.sum())
    degen_area = (art.surface_anon_ctf.mesh.area_faces < 1e-12)
    degen_pct = 100.0 * float(degen_area.sum()) / max(1, len(degen_area))
    pct_removed = 100.0 * (n_head - n_anon) / max(1, n_head)

    # Bidirectional vertex-identity sweep (§4.4.2 of the thesis).
    # Both meshes are in the CTF frame.
    V_in = np.asarray(art.surface_ctf.mesh.vertices)
    V_out = np.asarray(art.surface_anon_ctf.mesh.vertices)
    F_in = np.asarray(art.surface_ctf.mesh.faces)
    mask = art.mask

    # Forward: every anonymized vertex must coincide with an input vertex.
    tree_in = KDTree(V_in)
    fwd_d, _ = tree_in.query(V_out, k=1)
    fwd_max = float(fwd_d.max()) if len(fwd_d) else 0.0
    fwd_mismatches = int((fwd_d > 1e-9).sum())

    # Reverse: every expected-survivor input vertex must show up in the
    # anonymized output. An expected survivor is non-masked AND incident
    # to at least one face whose three vertices are all non-masked.
    face_all_kept = (~mask[F_in]).all(axis=1)
    incident_to_kept_face = np.zeros(len(V_in), dtype=bool)
    incident_to_kept_face[F_in[face_all_kept].ravel()] = True
    expected = (~mask) & incident_to_kept_face
    n_expected = int(expected.sum())
    if n_expected and len(V_out):
        tree_out = KDTree(V_out)
        rev_d, _ = tree_out.query(V_in[expected], k=1)
        rev_max = float(rev_d.max())
        rev_mismatches = int((rev_d > 1e-9).sum())
    else:
        rev_max = 0.0
        rev_mismatches = 0

    # Boundary orphans: non-masked vertices whose every incident face
    # was discarded because at least one of its corners is masked.
    has_kept_face = np.zeros(len(V_in), dtype=bool)
    has_kept_face[F_in[face_all_kept].ravel()] = True
    boundary_orphans = int(((~mask) & (~has_kept_face)).sum())

    row = {
        's_id': s_id(n),
        'subject': n,
        'optode': is_optode(n),
        'n_vertices_raw': surface_raw.nvertices,
        'n_faces_raw': surface_raw.nfaces,
        'n_vertices_head': n_head,
        'n_faces_head': n_faces_head,
        'mask_size': mask_size,
        'n_vertices_anonymized': n_anon,
        'n_faces_anonymized': n_faces_anon,
        'vertices_removed': int(n_head - n_anon),
        'faces_removed': int(n_faces_head - n_faces_anon),
        'pct_vertices_removed': pct_removed,
        'degenerate_face_pct': degen_pct,
        'cap_z_mm': art.cap_z,
        'forward_max_dist_mm': fwd_max,
        'forward_mismatches': fwd_mismatches,
        'reverse_max_dist_mm': rev_max,
        'reverse_mismatches': rev_mismatches,
        'boundary_orphans': boundary_orphans,
    }
    rows.append(row)
    print(
        f'  head: {n_head:,} v / {n_faces_head:,} f  ->  '
        f'anon: {n_anon:,} v / {n_faces_anon:,} f  '
        f'(-{pct_removed:.1f}%, degen {degen_pct:.3f}%, '
        f'cap_z {art.cap_z:.1f} mm)'
    )
    print(
        f'  sweep: fwd_max={fwd_max:.3g} mm, fwd_mismatches={fwd_mismatches}, '
        f'rev_max={rev_max:.3g} mm, rev_mismatches={rev_mismatches}, '
        f'boundary_orphans={boundary_orphans}'
    )

--- S1 (Subject16) ---


  head: 462,773 v / 816,754 f  ->  anon: 391,653 v / 684,912 f  (-15.4%, degen 0.000%, cap_z 10.0 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=58
--- S2 (Subject17) ---


  head: 678,081 v / 1,171,705 f  ->  anon: 557,573 v / 957,557 f  (-17.8%, degen 0.000%, cap_z 38.5 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=118
--- S3 (Subject18) ---


  head: 669,502 v / 1,160,366 f  ->  anon: 590,944 v / 1,021,132 f  (-11.7%, degen 0.000%, cap_z 10.0 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=131
--- S4 (Subject19) ---


  head: 408,990 v / 721,382 f  ->  anon: 306,641 v / 543,001 f  (-25.0%, degen 0.000%, cap_z 38.5 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=110
--- S5 (Subject20) ---


...

  head: 508,095 v / 909,530 f  ->  anon: 423,901 v / 754,775 f  (-16.6%, degen 0.000%, cap_z 10.0 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=74
--- S8 (Subject12) ---


  head: 372,526 v / 704,316 f  ->  anon: 289,640 v / 550,170 f  (-22.2%, degen 0.000%, cap_z 10.0 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=53
--- S9 (Subject13) ---


  head: 695,675 v / 1,279,792 f  ->  anon: 594,565 v / 1,093,389 f  (-14.5%, degen 0.000%, cap_z 23.5 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=69
--- S10 (Subject14) ---


  head: 386,762 v / 735,380 f  ->  anon: 291,130 v / 557,239 f  (-24.7%, degen 0.000%, cap_z 37.5 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=55
--- S11 (Subject15) ---


  head: 402,064 v / 765,393 f  ->  anon: 263,223 v / 506,223 f  (-34.5%, degen 0.000%, cap_z 33.5 mm)
  sweep: fwd_max=0 mm, fwd_mismatches=0, rev_max=0 mm, rev_mismatches=0, boundary_orphans=55


## 3. Summary table

One row per subject. Columns feed the mesh-statistics table and the mesh-integrity prose of Chapter 4.

In [4]:
df = pd.DataFrame(rows)
if len(df):
    df = df.iloc[df['s_id'].map(lambda s: int(s[1:])).argsort()].reset_index(drop=True)
df

,s_id,subject,optode,n_vertices_raw,n_faces_raw,n_vertices_head,n_faces_head,mask_size,n_vertices_anonymized,n_faces_anonymized,vertices_removed,faces_removed,pct_vertices_removed,degenerate_face_pct,cap_z_mm,forward_max_dist_mm,forward_mismatches,reverse_max_dist_mm,reverse_mismatches,boundary_orphans
0,S1,16,True,462773,816754,462773,816754,71062,391653,684912,71120,131842,15.368226,0.000000,10.0,0.0,0,0.0,0,58
1,S2,17,True,1284667,2223716,678081,1171705,120390,557573,957557,120508,214148,17.771918,0.000000,38.5,0.0,0,0.0,0,118
2,S3,18,True,1253509,2196208,669502,1160366,78427,590944,1021132,78558,139234,11.733796,0.000196,10.0,0.0,0,0.0,0,131
3,S4,19,True,408990,721382,408990,721382,102239,306641,543001,102349,178381,25.024817,0.000000,38.5,0.0,0,0.0,0,110
4,S5,20,True,576147,959550,576147,959550,173520,402437,677380,173710,282170,30.150292,0.000591,38.5,0.0,0,0.0,0,190
5,S6,21,True,494109,866676,494109,866676,88838,405221,704159,88888,162517,17.989553,0.000000,10.0,0.0,0,0.0,0,50
6,S7,22,True,508095,909530,508095,909530,84120,423901,754775,84194,154755,16.570523,0.000000,10.0,0.0,0,0.0,0,74
7,S8,12,False,624121,1169802,372526,704316,82833,289640,550170,82886,154146,22.249722,0.000000,10.0,0.0,0,0.0,0,53
8,S9,13,False,943539,1736738,695675,1279792,101041,594565,1093389,101110,186403,14.534086,0.000000,23.5,0.0,0,0.0,0,69
9,S10,14,False,725532,1370614,386762,735380,95577,291130,557239,95632,178141,24.726317,0.000000,37.5,0.0,0,0.0,0,55


## 4. Cohort-level numbers for the chapter prose

The mesh-integrity section cites cohort-level min/max/median degenerate-face percentage; the mesh-statistics section cites the overall vertex-removal range. Compute both here so the thesis text can just quote them.

In [5]:
if len(df):
    print(f'pct_vertices_removed: min={df.pct_vertices_removed.min():.2f}, '
          f'median={df.pct_vertices_removed.median():.2f}, '
          f'max={df.pct_vertices_removed.max():.2f}')
    print(f'cap_z_mm: min={df.cap_z_mm.min():.1f}, '
          f'median={df.cap_z_mm.median():.1f}, '
          f'max={df.cap_z_mm.max():.1f}')
    print(f'degenerate_face_pct: min={df.degenerate_face_pct.min():.3f}, '
          f'median={df.degenerate_face_pct.median():.3f}, '
          f'max={df.degenerate_face_pct.max():.3f}')
    print(f'forward_max_dist_mm:  max={df.forward_max_dist_mm.max():.3g}, '
          f'mismatches sum={int(df.forward_mismatches.sum())}')
    print(f'reverse_max_dist_mm:  max={df.reverse_max_dist_mm.max():.3g}, '
          f'mismatches sum={int(df.reverse_mismatches.sum())}')
    print(f'boundary_orphans: total={int(df.boundary_orphans.sum())}, '
          f'per-subject min={int(df.boundary_orphans.min())}, '
          f'max={int(df.boundary_orphans.max())}, '
          f'median={int(df.boundary_orphans.median())}')

pct_vertices_removed: min=11.73, median=17.99, max=34.53
cap_z_mm: min=10.0, median=23.5, max=38.5
degenerate_face_pct: min=0.000, median=0.000, max=0.001
forward_max_dist_mm:  max=0, mismatches sum=0
reverse_max_dist_mm:  max=0, mismatches sum=0
boundary_orphans: total=963, per-subject min=50, max=190, median=69


## 5. Save CSV

In [6]:
out = OUT_DIR / 'batch_validation.csv'

df.to_csv(out, index=False)

print(f'Wrote {out} ({len(df)} rows)')

Wrote thesis_results_out/batch_validation.csv (11 rows)
